# Load RBSA Site Metadata

This notebook loads the HEMS-RBSA building metadata through the existing `RBSALoader`, producing a list of validated `RBSASite` objects. These objects are the clean starting point for mapping survey fields into BEM input fields.

## Setup

Resolve the project paths and import the RBSA loader from `validation/src`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

src_candidates = [
    Path("../src").resolve(),
    Path("validation/src").resolve(),
]
SRC_DIR = next(path for path in src_candidates if (path / "rbsa_loader.py").exists())
DATA_DIR = SRC_DIR / "data" / "HEMS_RBSA"
BEM_DIR = SRC_DIR / "data" / "BEM"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rbsa_loader import RBSALoader

print(f"SRC_DIR:  {SRC_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

## Load RBSA Sites

`load_all()` reads the HEMS-RBSA CSV files, applies the existing key-column validity filter, deduplicates by `ee_site_id`, and returns validated `RBSASite` objects.

In [ ]:
loader = RBSALoader(DATA_DIR)
rbsa_sites = loader.load_all()

print(f"Loaded {len(rbsa_sites):,} RBSASite objects")

## Quick Preview

Flatten a few high-value fields so we can inspect whether the loader has the building geometry, envelope, HVAC, and DHW information we need.

In [ ]:
preview_rows = []
for site in rbsa_sites:
    primary_dhw = site.dhw.units[0] if site.dhw.units else None
    preview_rows.append({
        "site_id": site.info.site_id,
        "ee_site_id": site.info.ee_site_id,
        "state": site.info.state,
        "conditioned_area": site.geometry.conditioned_area,
        "conditioned_volume": site.geometry.conditioned_volume,
        "volume_to_area_ratio": site.geometry.conditioned_volume_to_area_ratio,
        "wwr": site.geometry.window_to_wall_ratio,
        "wall_u_value": site.envelope.total_wall_u_value,
        "ceiling_u_value": site.envelope.total_ceiling_u_value,
        "floor_u_value": site.envelope.total_floor_u_value,
        "window_u_value": site.envelope.window_u_value,
        "heating_system": site.hvac.primary_heating_system_type,
        "heating_fuel": site.hvac.primary_heating_fuel_type,
        "cooling_system": site.hvac.primary_cooling_system_type,
        "dhw_fuel": primary_dhw.fuel_type if primary_dhw else None,
        "dhw_efficiency": primary_dhw.efficiency if primary_dhw else None,
    })

rbsa_site_preview = pd.DataFrame(preview_rows)
display(rbsa_site_preview.head(20))

## Save Normalized Metadata

Optionally refresh the normalized JSON copy used elsewhere in the project.

In [ ]:
SAVE_NORMALIZED_JSON = False
NORMALIZED_JSON_PATH = BEM_DIR / "rbsa_sites.json"

if SAVE_NORMALIZED_JSON:
    RBSALoader.save(rbsa_sites, NORMALIZED_JSON_PATH)
else:
    print(f"Not saving. Set SAVE_NORMALIZED_JSON = True to write {NORMALIZED_JSON_PATH}")